# ViHSD - Baseline XLM-RoBERTa + ViCLSR

Notebook Kaggle chạy hai baseline `FacebookAI/xlm-roberta-base` và `huynhtin/ViCLSR`.

Bật Internet và GPU T4 trước khi chạy. ViCLSR dùng XLM-RoBERTa-Large nên chậm và tốn VRAM hơn XLM-RoBERTa-base.

In [ ]:
import os
import subprocess
from pathlib import Path

EXP_REL = Path("notebooks/models/baselines/ViHSD - Baseline XLM-RoBERTa_ViCLSR")
ROOT = Path.cwd()

# If this notebook is opened outside the repo, clone the project first.
if not (ROOT / EXP_REL / "run_two_models.py").exists():
    if not (ROOT / "ViAmpleHate").exists():
        subprocess.run(["git", "clone", "-b", "trung-dev", "https://github.com/MinhTuan2405/ViAmpleHate.git"], check=True)
    os.chdir(ROOT / "ViAmpleHate")

ROOT = Path.cwd()
EXP_DIR = ROOT / EXP_REL
SCRIPT = str(EXP_DIR / "run_two_models.py")
REQS = str(EXP_DIR / "requirements.txt")

assert Path(SCRIPT).exists(), f"Không tìm thấy {SCRIPT}"
assert Path(REQS).exists(), f"Không tìm thấy {REQS}"
print("Repo path:", ROOT)
print("Baseline path:", EXP_DIR)


In [ ]:
!pip install -q -r "{REQS}"

## Cấu hình

Notebook này cố định chạy `ViHSD`. Dùng các split train/validation/test có sẵn và map về hai nhãn NON-HATE/HATE.

In [ ]:
DATASET = "vihsd"
EPOCHS = 5
MAX_LEN = 128
OUTPUT_DIR = "/kaggle/working/viamplehate_runs_vihsd_seed42"

# Khớp cách lấy mẫu trong các notebook baseline VOZ-HSD của repo.
VOZ_SPLIT_POLICY = "baseline"
VOZ_SAMPLE_SIZE = 100_000
VOZ_HATE_RATIO = 0.10  # Chỉ được dùng khi policy="proposed".


## Smoke Test

Chạy 2 cell này trước để kiểm tra path, dataset, model loading, forward/backward, save metrics. XLM-RoBERTa smoke test nhanh; ViCLSR smoke test vẫn phải tải checkpoint lớn lần đầu.

In [ ]:
!python "{SCRIPT}" \
  --dataset {DATASET} \
  --model xlm-roberta \
  --epochs 1 \
  --max-len 64 \
  --batch-size 2 \
  --eval-batch-size 4 \
  --voz-split-policy {VOZ_SPLIT_POLICY} \
  --voz-sample-size {VOZ_SAMPLE_SIZE} \
  --voz-hate-ratio {VOZ_HATE_RATIO} \
  --output-dir /kaggle/working/viamplehate_smoke \
  --smoke-test

In [ ]:
!python "{SCRIPT}" \
  --dataset {DATASET} \
  --model viclsr \
  --epochs 1 \
  --max-len 64 \
  --batch-size 1 \
  --eval-batch-size 2 \
  --voz-split-policy {VOZ_SPLIT_POLICY} \
  --voz-sample-size {VOZ_SAMPLE_SIZE} \
  --voz-hate-ratio {VOZ_HATE_RATIO} \
  --output-dir /kaggle/working/viamplehate_smoke \
  --smoke-test

## 1. Chạy XLM-RoBERTa

Model này nhẹ hơn ViCLSR vì dùng `xlm-roberta-base`. Nếu GPU yếu, giảm `--batch-size 4`.

In [ ]:
!python "{SCRIPT}" \
  --dataset {DATASET} \
  --model xlm-roberta \
  --epochs {EPOCHS} \
  --max-len {MAX_LEN} \
  --batch-size 8 \
  --eval-batch-size 16 \
  --voz-split-policy {VOZ_SPLIT_POLICY} \
  --voz-sample-size {VOZ_SAMPLE_SIZE} \
  --voz-hate-ratio {VOZ_HATE_RATIO} \
  --output-dir {OUTPUT_DIR}

## 2. Chạy ViCLSR

ViCLSR là XLM-RoBERTa-Large và checkpoint khoảng vài GB. Cell này dùng batch nhỏ. Nếu CUDA OOM, đổi `--max-len 128` hoặc giữ `--batch-size 1`.

In [ ]:
!python "{SCRIPT}" \
  --dataset {DATASET} \
  --model viclsr \
  --epochs {EPOCHS} \
  --max-len {MAX_LEN} \
  --batch-size 1 \
  --eval-batch-size 2 \
  --voz-split-policy {VOZ_SPLIT_POLICY} \
  --voz-sample-size {VOZ_SAMPLE_SIZE} \
  --voz-hate-ratio {VOZ_HATE_RATIO} \
  --output-dir {OUTPUT_DIR}

## 3. Xem metrics

Mỗi model lưu `best_model.pt`, `metrics.json`, và tokenizer vào `/kaggle/working`.

In [ ]:
import json
from pathlib import Path

base = Path(OUTPUT_DIR) / DATASET
for model_name in ["xlm-roberta", "viclsr"]:
    path = base / model_name / "metrics.json"
    print("\n===", model_name, "===")
    if not path.exists():
        print("Chưa có metrics:", path)
        continue
    metrics = json.loads(path.read_text(encoding="utf-8"))
    print({k: metrics[k] for k in ["accuracy", "macro_f1", "hate_f1"]})
    print(metrics["report"])

## Ghi chú báo cáo

Khi viết report, ưu tiên so sánh `macro_f1` và `hate_f1`, vì class `HATE` ít hơn nhiều so với `NON-HATE`.